# Notebook 07: Cross-Condition Analysis

**Agency Calculus Empirical Validation — Paper C**

This notebook:
1. Loads all training results (SUM, NASH, JAM, 5 seeds each)
2. Aggregates across seeds (mean ± std)
3. Generates all paper-quality figures
4. Computes statistical tests
5. Produces the summary table

Run this after all training runs in notebooks 04-06 are complete.

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────
# If ai_economist is missing, run notebook 01 first (it handles installation
# and the required kernel restart).
import sys, os

try:
    import ai_economist  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        "\n❌  ai_economist not found. Run notebook 01_setup_and_test first,\n"
        "    restart the kernel, then return here."
    )

# Add src/ to path
for candidate in [
    '/content/ac-validation/src',
    os.path.join(os.getcwd(), '..', 'src'),
    os.path.join(os.getcwd(), 'src'),
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        print(f'src on path: {candidate}')
        break


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from metrics import MetricsLogger
from plotting import (
    plot_main_dashboard, plot_final_values_bar, plot_poli_radar,
    merge_seed_results, CONDITION_COLORS, CONDITION_LABELS, METRIC_LABELS,
)

RESULTS_DIR = '../results'
FIGURES_DIR = '../results/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

CONDITIONS = ['sum', 'nash', 'jam']
N_SEEDS = 5
print(f'Results directory: {RESULTS_DIR}')

## 1. Load All Results

In [ ]:
# Load all available seed results
all_loggers = {}   # condition -> list of MetricsLogger
missing = []

for condition in CONDITIONS:
    loggers = []
    for seed in range(N_SEEDS):
        path = f'{RESULTS_DIR}/{condition}_seed{seed}_metrics.npz'
        if os.path.exists(path):
            loggers.append(MetricsLogger.load(path))
        else:
            missing.append(f'{condition}_seed{seed}')
    all_loggers[condition] = loggers
    print(f'{condition.upper()}: {len(loggers)}/{N_SEEDS} seeds loaded')

if missing:
    print(f'\nMissing: {missing}')
    print('Run notebooks 04-06 to generate missing results.')
else:
    print('\nAll results loaded!')

In [ ]:
# Build merged data dict for plotting
# Format: condition -> {metric: mean_array, metric_std: std_array, steps: array}

MAIN_METRICS = [
    'floor_utility', 'total_utility', 'gini_wealth',
    'floor_agency', 'floor_tax_rate', 'fhi',
    'resource_concentration', 'floor_action_diversity',
]

plot_data = {}
final_values = {}

for condition, loggers in all_loggers.items():
    if not loggers:
        continue
    
    condition_data = {}
    condition_final = {}
    
    for metric in MAIN_METRICS:
        # Try aggregated metric keys (with _mean suffix from record_aggregated)
        metric_key = f'{metric}_mean'
        steps, mean, std = merge_seed_results(loggers, metric_key)
        if len(steps) == 0:
            # Try plain key
            steps, mean, std = merge_seed_results(loggers, metric)
        if len(steps) > 0:
            condition_data['steps'] = steps
            condition_data[metric] = mean
            condition_data[f'{metric}_std'] = std
            condition_final[metric] = float(mean[-5:].mean()) if len(mean) >= 5 else float(mean[-1])
    
    plot_data[condition] = condition_data
    final_values[condition] = condition_final

available = [c for c in CONDITIONS if c in plot_data and plot_data[c]]
print(f'Conditions with data: {available}')

## 2. Main Dashboard

In [ ]:
if available:
    fig = plot_main_dashboard(
        {c: plot_data[c] for c in available},
        save_path=f'{FIGURES_DIR}/main_dashboard.png',
    )
    plt.show()
else:
    print('No data yet. Run training notebooks first.')
    print('Showing placeholder with synthetic data...')
    
    # Generate synthetic demonstration data
    steps = np.arange(0, 10_000_001, 100_000)
    synthetic = {
        'sum': {
            'steps': steps,
            'floor_utility': 10 - 3 * (1 - np.exp(-steps / 5e6)) + np.random.randn(len(steps)) * 0.3,
            'total_utility': 25 + 15 * (1 - np.exp(-steps / 3e6)) + np.random.randn(len(steps)) * 0.5,
            'gini_wealth': 0.2 + 0.3 * (1 - np.exp(-steps / 4e6)) + np.random.randn(len(steps)) * 0.02,
            'floor_agency': 0.3 - 0.15 * (1 - np.exp(-steps / 5e6)) + np.random.randn(len(steps)) * 0.02,
            'floor_tax_rate': 0.1 + 0.25 * (1 - np.exp(-steps / 3e6)) + np.random.randn(len(steps)) * 0.02,
            'fhi': 0.1 + 0.4 * (1 - np.exp(-steps / 4e6)) + np.random.randn(len(steps)) * 0.03,
        },
        'nash': {
            'steps': steps,
            'floor_utility': 10 + 2 * (1 - np.exp(-steps / 5e6)) + np.random.randn(len(steps)) * 0.3,
            'total_utility': 25 + 10 * (1 - np.exp(-steps / 3e6)) + np.random.randn(len(steps)) * 0.5,
            'gini_wealth': 0.2 + 0.1 * (1 - np.exp(-steps / 4e6)) + np.random.randn(len(steps)) * 0.02,
            'floor_agency': 0.3 + 0.05 * (1 - np.exp(-steps / 5e6)) + np.random.randn(len(steps)) * 0.02,
            'floor_tax_rate': 0.1 + 0.05 * (1 - np.exp(-steps / 3e6)) + np.random.randn(len(steps)) * 0.02,
            'fhi': 0.1 + 0.15 * (1 - np.exp(-steps / 4e6)) + np.random.randn(len(steps)) * 0.03,
        },
        'jam': {
            'steps': steps,
            'floor_utility': 10 + 8 * (1 - np.exp(-steps / 5e6)) + np.random.randn(len(steps)) * 0.3,
            'total_utility': 25 + 5 * (1 - np.exp(-steps / 3e6)) + np.random.randn(len(steps)) * 0.5,
            'gini_wealth': 0.2 - 0.05 * (1 - np.exp(-steps / 4e6)) + np.random.randn(len(steps)) * 0.02,
            'floor_agency': 0.3 + 0.25 * (1 - np.exp(-steps / 5e6)) + np.random.randn(len(steps)) * 0.02,
            'floor_tax_rate': 0.1 - 0.05 * (1 - np.exp(-steps / 3e6)) + np.random.randn(len(steps)) * 0.02,
            'fhi': 0.1 - 0.05 * (1 - np.exp(-steps / 4e6)) + np.random.randn(len(steps)) * 0.03,
        },
    }
    fig = plot_main_dashboard(synthetic,
        save_path=f'{FIGURES_DIR}/main_dashboard_synthetic.png')
    plt.show()
    print('Note: SYNTHETIC DATA — replace with real results after training')

## 3. Final Values Bar Chart

In [ ]:
if final_values:
    fig = plot_final_values_bar(
        final_values,
        save_path=f'{FIGURES_DIR}/final_values_bar.png',
    )
    plt.show()

## 4. Statistical Tests

Mann-Whitney U test (non-parametric) comparing final metric values
across seed distributions.

In [ ]:
def get_final_seed_values(loggers, metric):
    """Get final (last eval) value for each seed."""
    values = []
    for logger in loggers:
        arrays = logger.to_arrays()
        key = f'{metric}_mean'
        if key not in arrays:
            key = metric
        if key in arrays and len(arrays[key]) > 0:
            # Mean of last 5 checkpoints for stability
            v = arrays[key]
            values.append(float(v[-5:].mean() if len(v) >= 5 else v[-1]))
    return values

key_metrics = ['floor_utility', 'total_utility', 'gini_wealth', 'floor_agency']
comparisons = [('jam', 'sum'), ('jam', 'nash'), ('nash', 'sum')]

print('Statistical Tests (Mann-Whitney U, two-sided)')
print('='*70)

rows = []
for metric in key_metrics:
    for cond_a, cond_b in comparisons:
        vals_a = get_final_seed_values(all_loggers.get(cond_a, []), metric)
        vals_b = get_final_seed_values(all_loggers.get(cond_b, []), metric)
        
        if len(vals_a) >= 2 and len(vals_b) >= 2:
            stat, pval = stats.mannwhitneyu(vals_a, vals_b, alternative='two-sided')
            mean_a = np.mean(vals_a)
            mean_b = np.mean(vals_b)
            sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
            rows.append({
                'Metric': METRIC_LABELS.get(metric, metric),
                'A': cond_a.upper(),
                'B': cond_b.upper(),
                f'Mean {cond_a.upper()}': f'{mean_a:.3f}',
                f'Mean {cond_b.upper()}': f'{mean_b:.3f}',
                'p-value': f'{pval:.4f}',
                'Sig': sig,
            })
        else:
            rows.append({
                'Metric': metric,
                'A': cond_a.upper(),
                'B': cond_b.upper(),
                f'Mean {cond_a.upper()}': 'N/A',
                f'Mean {cond_b.upper()}': 'N/A',
                'p-value': 'N/A',
                'Sig': 'N/A (insufficient data)',
            })

if rows:
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    df.to_csv(f'{RESULTS_DIR}/statistical_tests.csv', index=False)
    print(f'\nSaved to {RESULTS_DIR}/statistical_tests.csv')
else:
    print('No data available for statistical testing.')

## 5. Summary Table

In [ ]:
print('Summary Table: Final Metric Values by Condition')
print('='*70)
print(f'{"Metric":<35} {"SUM":>12} {"NASH":>12} {"JAM":>12}')
print('-'*70)

for metric in key_metrics + ['resource_concentration', 'floor_tax_rate']:
    row_vals = {}
    for condition in CONDITIONS:
        vals = get_final_seed_values(all_loggers.get(condition, []), metric)
        if vals:
            row_vals[condition] = f'{np.mean(vals):.3f} ± {np.std(vals):.3f}'
        else:
            row_vals[condition] = 'N/A'
    
    label = METRIC_LABELS.get(metric, metric)[:34]
    print(f'{label:<35} {row_vals.get("sum", "N/A"):>12} '
          f'{row_vals.get("nash", "N/A"):>12} {row_vals.get("jam", "N/A"):>12}')

print('='*70)
print('\nExpected pattern: JAM lowest total utility, highest floor utility')
print('This is the compensation mechanism — the critical theoretical prediction.')

## 6. POLI Radar Chart

In [ ]:
# Extract POLI dimension means from logged agency data
# (requires poli dimension logging in training — use available data)

poli_dims = ['prerequisites', 'options', 'levers', 'impact', 'knowledge']
poli_means = {}

for condition, loggers in all_loggers.items():
    if not loggers:
        continue
    dim_vals = {d: [] for d in poli_dims}
    for logger in loggers:
        arrays = logger.to_arrays()
        for dim in poli_dims:
            key = f'floor_{dim}_mean'
            if key in arrays and len(arrays[key]) > 0:
                dim_vals[dim].append(float(arrays[key][-1]))
    
    if any(dim_vals[d] for d in poli_dims):
        poli_means[condition] = {d: np.mean(v) if v else 0.3 for d, v in dim_vals.items()}

if poli_means:
    from plotting import plot_poli_radar
    fig = plot_poli_radar(poli_means,
                          save_path=f'{FIGURES_DIR}/poli_radar.png')
    plt.show()
else:
    print('POLI dimension data not logged separately.')
    print('Agency scores are available — see floor_agency metric.')

## 7. The Critical Result

The test of the theory is the **floor-total tradeoff:**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for condition in CONDITIONS:
    floor_vals = get_final_seed_values(all_loggers.get(condition, []), 'floor_utility')
    total_vals = get_final_seed_values(all_loggers.get(condition, []), 'total_utility')
    
    if floor_vals and total_vals:
        ax.scatter(np.mean(total_vals), np.mean(floor_vals),
                   color=CONDITION_COLORS[condition], s=200, zorder=5,
                   label=CONDITION_LABELS[condition])
        # Error bars
        ax.errorbar(np.mean(total_vals), np.mean(floor_vals),
                    xerr=np.std(total_vals), yerr=np.std(floor_vals),
                    color=CONDITION_COLORS[condition], alpha=0.5, capsize=5)
        ax.annotate(condition.upper(),
                    (np.mean(total_vals), np.mean(floor_vals)),
                    textcoords='offset points', xytext=(10, 5), fontsize=11)

ax.set_xlabel('Total System Utility Σu_i', fontsize=12)
ax.set_ylabel('Floor Agent Utility min(u_i)', fontsize=12)
ax.set_title(
    'The Compensation Tradeoff: Total Utility vs Floor Utility\n'
    'Prediction: JAM = lower total, higher floor (upper-left quadrant from SUM)',
    fontsize=11
)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/critical_tradeoff.png', bbox_inches='tight', dpi=150)
plt.show()

print('Theory predicts:')
print('  JAM: high floor utility, lower total utility (upper-left of SUM)')
print('  SUM: low floor utility, high total utility (lower-right)')
print('  NASH: intermediate (between JAM and SUM)')